In [0]:
from pyspark.sql import functions as F

bronze_path = "/Volumes/workspace/default/literatura_bronze/query1.csv"

df_bronze = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(bronze_path)
)

display(df_bronze)

In [0]:
bronze_table = "workspace.default.bronze_wikidata_authors_raw"

(
    df_bronze.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(bronze_table)
)

print(f"Tabela criada: {bronze_table}")

In [0]:
display(
    spark.sql("""
        SELECT *
        FROM workspace.default.bronze_wikidata_authors_raw
        LIMIT 10
    """)
)

In [0]:
spark.sql("""
    SELECT COUNT(*) AS total_rows
    FROM workspace.default.bronze_wikidata_authors_raw
""").show()

In [0]:
import json
from urllib.request import Request, urlopen

test_url = "https://openlibrary.org/authors/OL1527013A.json"

request = Request(
    test_url,
    headers={
        "User-Agent": "mvp-literatura-sul-global-databricks/1.0 (academic data pipeline)"
    },
)

with urlopen(request, timeout=30) as response:
    author_test = json.loads(response.read().decode("utf-8"))

display(
    spark.createDataFrame(
        [
            (
                test_url,
                author_test.get("name"),
                author_test.get("key"),
            )
        ],
        ["source_url", "author_name", "author_key"],
    )
)

In [0]:
import json
import time
from datetime import datetime, timezone
from urllib.request import Request, urlopen

openlibrary_author_ids = [
    "OL1527013A",
    "OL12217460A",
    "OL6495716A",
    "OL5697446A",
    "OL8701565A",
    "OL5122048A",
    "OL2940871A",
    "OL7633883A",
    "OL9081057A",
    "OL13513413A",
    "OL11730977A",
    "OL10432462A",
    "OL5533173A",
    "OL718270A",
]

def get_json(url):
    request = Request(
        url,
        headers={
            "User-Agent": "mvp-literatura-sul-global-databricks/1.0 (academic data pipeline)"
        },
    )
    with urlopen(request, timeout=30) as response:
        return json.loads(response.read().decode("utf-8"))

extracted_at_utc = datetime.now(timezone.utc).isoformat()
author_records = []
works_records = []

for author_id in openlibrary_author_ids:
    author_url = f"https://openlibrary.org/authors/{author_id}.json"
    works_url = f"https://openlibrary.org/authors/{author_id}/works.json?limit=100"

    try:
        author_json = get_json(author_url)
        works_json = get_json(works_url)

        author_records.append({
            "source_author_id": author_id,
            "source_url": author_url,
            "extracted_at_utc": extracted_at_utc,
            "raw_json": json.dumps(author_json, ensure_ascii=False),
        })

        works_records.append({
            "source_author_id": author_id,
            "source_url": works_url,
            "extracted_at_utc": extracted_at_utc,
            "raw_json": json.dumps(works_json, ensure_ascii=False),
        })

        print(f"OK: {author_id}")

    except Exception as error:
        print(f"ERRO: {author_id} - {error}")

    time.sleep(1)

print(f"Autores extraídos: {len(author_records)}")
print(f"Respostas de obras extraídas: {len(works_records)}")

In [0]:
from pyspark.sql import Row

df_openlibrary_authors_raw = spark.createDataFrame(
    [Row(**record) for record in author_records]
)

df_openlibrary_works_raw = spark.createDataFrame(
    [Row(**record) for record in works_records]
)

(
    df_openlibrary_authors_raw.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.bronze_openlibrary_authors_raw")
)

(
    df_openlibrary_works_raw.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.bronze_openlibrary_works_raw")
)

print("Tabelas Bronze Open Library criadas com sucesso.")

In [0]:
display(
    spark.sql("""
        SELECT
            'bronze_wikidata_authors_raw' AS table_name,
            COUNT(*) AS total_rows
        FROM workspace.default.bronze_wikidata_authors_raw

        UNION ALL

        SELECT
            'bronze_openlibrary_authors_raw' AS table_name,
            COUNT(*) AS total_rows
        FROM workspace.default.bronze_openlibrary_authors_raw

        UNION ALL

        SELECT
            'bronze_openlibrary_works_raw' AS table_name,
            COUNT(*) AS total_rows
        FROM workspace.default.bronze_openlibrary_works_raw
    """)
)

In [0]:
display(
    spark.sql("""
        SELECT
            source_author_id,
            source_url,
            extracted_at_utc,
            substring(raw_json, 1, 500) AS raw_json_preview
        FROM workspace.default.bronze_openlibrary_works_raw
        ORDER BY source_author_id
        LIMIT 5
    """)
)